# LLM Zoomcamp - Agentic RAG Homework

Building a complete RAG pipeline using MinSearch, OpenAI, Chunking and ToyAIKit.

In [2]:
#import libraries

from ingest import load_data
from minsearch import Index

from openai import OpenAI
from dotenv import load_dotenv

## Setup and Data Loading

Load API keys and GitHub lesson documents.

In [3]:
#load the api key and git hub documents
load_dotenv()
openai_client = OpenAI()
documents = load_data()

## Q1 - Count Lesson Pages

Load all lesson documents and verify the dataset size.

In [ ]:
len(documents)

## Q2 - Indexing and Searching

Create a MinSearch index and search lesson content.

In [4]:
# first create the index

index_search = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index_search.fit(documents)

In [ ]:
#Create the Search Function

def search(query: str) -> dict[str, str]:
    
    """
        Search the index for entires matching given query
    """

    boost_dict = {'content':3.0}

    return index_search.search(
        query,
        boost_dict=boost_dict,
        num_results=5
    )

In [ ]:
question = search('How does the agentic loop keep calling the model until it stops?')
question

## Q3 - Build Manual RAG

Convert retrieved documents into context and send them to GPT.

In [ ]:
# convert the document to llm level context

def build_context(documents):
    lines = []

    for doc in documents:
        lines.append("content: " + doc['content'])
        lines.append("finename: " + doc['filename'])
        lines.append("")

    return '\n'.join(lines).strip()

In [ ]:
# create the instructions and user_prompt
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

user_prompt = """
Question : 
{question}

Context:
{context}
"""

In [12]:
#create the prompt

def build_prompt(query, documents):
    context = build_context(documents)
    prompt = user_prompt.format(
        question=query,
        context =  context
    )

    return prompt.strip()
    

In [ ]:
# create our llm

def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    messages = [
        {'role':'developer', 'content':instructions},
        {'role':'user', 'content':user_prompt}
    ]

    response = openai_client.responses.create(
        input=messages,
        model=model
    )

    return response

In [14]:
#final create our rag

def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt_results = build_prompt(query, search_results)
    results = llm(instructions, prompt_results, model=model)

    return results

### Token Usage Analysis

Measure prompt token consumption for the RAG request.

In [ ]:
question = rag('How does the agentic loop keep calling the model until it stops?')
question.usage.input_tokens

## Q4 - Document Chunking

Split large documents into overlapping chunks.

In [30]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

## Q5 - RAG with Chunking

Build a new index using chunks and compare token usage.


In [ ]:
index_search.fit(chunks)
question.usage.input_tokens

## Q6 - Agentic RAG using ToyAIKit

Convert the search function into a tool and allow the model to decide when to search.

In [23]:
# For Q6 instead of our rag just create the search function and pass it to our framework and run the rag

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [28]:
def using_framework(query):

    agent_tool = Tools()
    agent_tool.add_tool(search)

    chat_interface = IPythonChatInterface()
    callback = DisplayingRunnerCallback(chat_interface)

    runner = OpenAIResponsesRunner(
        tools=agent_tool,
        developer_prompt=instructions,
        chat_interface=chat_interface,
        llm_client=OpenAIClient(model='gpt-5.4-mini')
    )


    result = runner.loop(
        prompt=query,
        callback=callback
    )

    return result

In [29]:
ans = using_framework('How does the agentic loop keep calling the model until it stops?')
ans

-> Response received


-> Response received


LoopResult(new_messages=[EasyInputMessage(content="\nYou're a course teaching assistant. \nAnswer the student's question using the search tool. \nMake multiple searches with different keywords before answering.\n", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop keep calling the model until it stops?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop keep calling the model until it stops how does it work"}', call_id='call_Kgr7hwiiS3iW12jKDZp4kaUZ', name='search', type='function_call', id='fc_08a73a414f9eb229006a30168e4454819da503712448ee0561', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop stops when model returns final answer"}', call_id='call_KsUglnoW7F23FdAhifa8ueJ5', name='search', type='function_call', id='fc_08a73a414f9eb229006a30168e4464819d9d253a593ae22755', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query"